# 🧠 NVIDIA Nemotron-3-Nano-30B Supervised Fine-Tuning (SFT) Notebook

This notebook contains a complete **Supervised Fine-Tuning (SFT)** pipeline to train a LoRA adapter on the `train.csv` reasoning dataset. 

### 🏆 Key Features of this Training Pipeline:
1. **Modern TRL v1.x API Integration**: Conforms to `trl>=1.0.0` standards using `SFTConfig` and native completion-only loss masking.
2. **Target-Only Loss Masking**: Uses `completion_only_loss=True` to automatically detect chat-template turns and mask prompt instruction loss.
3. **Parameter Efficient Fine-Tuning (PEFT)**: Sets up a rank 32 LoRA adapter targeting Nemotron projection layers.
4. **Kaggle GPU Optimized**: Configured for mixed-precision (`bfloat16`), gradient accumulation, and gradient checkpointing to fit training within standard Kaggle GPU limits (e.g., 2x T4 or L4/A100).
5. **Offline & Save Version Optimizations**: Configured to run offline without internet-based `pip` installations by loading pre-installed libraries, the NVIDIA Cutlass package path, and installing local wheels. Includes a `DRY_RUN` toggle to quickly save the notebook without running long training loops.

---

## 🛠️ Step 1: Offline Wheels Installation

In [ ]:
import os
import glob
import subprocess

# Install all local packages offline from the ml-wheels directory
wheels_dir = "/kaggle/input/datasets/ckskaggle/ml-wheels/"
if os.path.exists(wheels_dir):
    print(f"Found offline wheels directory: {wheels_dir}")
    wheel_files = glob.glob(os.path.join(wheels_dir, "*.whl"))
    if wheel_files:
        print(f"Found {len(wheel_files)} wheels. Installing offline...")
        # Install all wheels together so pip resolves local inter-dependencies correctly
        cmd = f"pip install --no-index --find-links={wheels_dir} " + " ".join(wheel_files)
        print(f"Running: {cmd}")
        try:
            subprocess.run(cmd, shell=True, check=True)
            print("All offline wheels installed successfully!")
        except Exception as e:
            print(f"Offline installation failed: {e}")
    else:
        print("No wheel (.whl) files found in the directory.")
else:
    print("Offline wheels directory not found. Assuming environment is already set up.")

## ⚙️ Step 2: Environment Configuration & Import Dependencies

Next, we configure NVIDIA utility directories and load dependencies.

In [ ]:
import os
import shutil
import stat
import glob
import site
import sys

# --- 1. CRITICAL NVIDIA / TRITON PTXAS PERMISSION PATCH ---
# The pre-installed 'ptxas-blackwell' binary in the utility dataset lacks execute permissions
# and sits in a read-only directory. We copy it to a writable '/tmp' folder and override paths.
print("Configuring Triton binary overrides...")
candidates = glob.glob("/kaggle/usr/lib/notebooks/*/nvidia*utility*script/triton/backends/nvidia/bin/ptxas-blackwell")
if candidates:
    src = candidates[0]
    dst_dir = "/tmp/triton-bin"
    os.makedirs(dst_dir, exist_ok=True)
    dst = f"{dst_dir}/ptxas-blackwell"
    
    # Copy to a writable directory
    shutil.copy2(src, dst)
    
    # Grant execute permissions
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    
    # Redirect Triton to the new location
    os.environ["TRITON_PTXAS_PATH"] = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst
    os.environ["PATH"] = f"{dst_dir}:" + os.environ["PATH"]
    
    # Blackwell environment flags
    os.environ["CUDA_MODULE_LOADING"] = "LAZY"
    os.environ["TRITON_DISABLE_AUTOTUNE"] = "1"
    os.environ["USE_TRITON"] = "0"
    os.environ["PTXAS_PATH"] = dst
    print(f"PTXAS override successful: {dst}")
else:
    print("ptxas-blackwell not found in utility scripts. Assuming other architecture or local run.")

# --- 2. CRITICAL CACHING DIRECTORY REDIRECTS ---
# Force all caching, JIT, torchao, and trl compilation to use the writable directory
print("Configuring caching directories...")
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["TORCH_EXTENSIONS_DIR"] = "/kaggle/working/torch_extensions"
os.environ["TRITON_CACHE_DIR"] = "/kaggle/working/triton_cache"

os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/torch_extensions", exist_ok=True)
os.makedirs("/kaggle/working/triton_cache", exist_ok=True)



In [ ]:
# --- 3. NVIDIA CUTLASS PATH CONFIGURATION ---
# Add NVIDIA Cutlass DSL package path (needed offline on Kaggle to load Mamba kernels)
import glob
import os
import site

# Dynamically search the usr/lib directory for the nvidia_cutlass_dsl package
search_path = "/kaggle/usr/lib/notebooks/*/nvidia*utility*script/nvidia_cutlass_dsl/python_packages/"
candidates = glob.glob(search_path)

if candidates:
    cutlass_pkg_path = candidates[0]
    site.addsitedir(cutlass_pkg_path)
    print(f"NVIDIA Cutlass DSL package path loaded successfully from: {cutlass_pkg_path}")
else:
    print("NVIDIA Cutlass package path not found automatically.")


In [ ]:
import torch
import polars as pl
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq
)
from trl import SFTTrainer, SFTConfig
import kagglehub

# Check CUDA
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# --- CONFIGURATION TOGGLES ---
# Set DRY_RUN = True to test notebook execution quickly (skips training & zipping)
# Set DRY_RUN = False to run full SFT training and output the final submission.zip
DRY_RUN = False

SHOULD_RUN_PIPELINE = not DRY_RUN
print(f"Should execute full training pipeline: {SHOULD_RUN_PIPELINE}")

## 📥 Step 3: Load and Format the Dataset

We load `train.csv` and format it. In this competition, the evaluation script extracts the answer from `\boxed{}`. Therefore, during SFT we must wrap our target answer in `\boxed{}`.

In [ ]:
# Load local data (make sure data path points to train.csv on Kaggle or local workspace)
data_path = "DATA/train.csv" if os.path.exists("DATA/train.csv") else "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pl.read_csv(data_path)

# --- LIMIT ROWS FOR FAST RUNS (Set to None for full training) ---
LIMIT_TRAIN_ROWS = 10
if LIMIT_TRAIN_ROWS is not None:
    df = df.head(LIMIT_TRAIN_ROWS)

print(f"Loaded {len(df)} training examples.")

# Convert polars dataframe to Hugging Face dataset
dataset = Dataset.from_pandas(df.to_pandas())

def format_chat_template(example):
    """
    Constructs standard chat dialogues.
    We wrap the answer in \boxed{} as required by the competition evaluation metric.
    """
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": f"\\boxed{{{example['answer']}}}"}
    ]
    return {"messages": messages}

# Apply standard formatting and remove original columns to avoid TRL namespace conflicts
# (If the 'prompt' column remains, SFTTrainer falsely assumes a legacy prompt-completion schema)
dataset = dataset.map(format_chat_template, remove_columns=dataset.column_names)
print("Sample Formatted Messages:")
print(dataset[0]["messages"])

## 🤖 Step 4: Load Model & Tokenizer

We download and load `NVIDIA Nemotron-3-Nano-30B-A3B-BF16`. We configure the model with `bfloat16` and gradient checkpointing to conserve memory.

In [ ]:
if SHOULD_RUN_PIPELINE:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    # Load tokenizer and set pad token
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
        use_cache=False  # Must be False when using gradient checkpointing
    )
    model.gradient_checkpointing_enable()
    print("Model and tokenizer loaded successfully.")
else:
    print("Skipping model and tokenizer loading (DRY_RUN).")

## 📐 Step 5: Configure LoRA Adapter

We setup `peft.LoraConfig` matching the allowed constraints (rank <= 32) and target modules. We define the configuration here and pass it to SFTTrainer which handles model wrapping automatically.

In [ ]:
if SHOULD_RUN_PIPELINE:
    LORA_RANK = 32

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=16,
        target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    print("LoRA config defined successfully.")
else:
    print("Skipping LoRA configuration (DRY_RUN).")

## 🏋️ Step 6: SFT Trainer Configuration

We use `SFTConfig` to configure hyperparameters and enable completion-only training natively via `completion_only_loss=True`. SFTTrainer parses the tokenizer's chat template structures and masks prompt tokens automatically.

In [ ]:
if SHOULD_RUN_PIPELINE:
    # Configure SFT Trainer parameters
    # Note: Modern TRL v1.5.1 uses 'max_length' instead of 'max_seq_length' inside SFTConfig
    training_args = SFTConfig(
        output_dir="./nemotron_adapter",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,  # Virtual batch size of 16
        learning_rate=1e-4,
        logging_steps=10,
        num_train_epochs=1,  # Adjust as needed (e.g. 1-3 epochs)
        bf16=True,
        optim="paged_adamw_8bit",  # Conserves VRAM
        save_strategy="no",        # Save manual adapter at the end
        report_to="none",
        max_length=1024,           # Truncate long contexts to save memory (replaced max_seq_length)
        completion_only_loss=True  # Automatically masks instruction/prompt loss natively!
    )

    # Initialize Trainer
    # We pass the base model and lora_config; SFTTrainer wraps the model automatically
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
        peft_config=lora_config
    )

    print("Starting model training...")
    trainer.train()
    print("Training completed!")
else:
    print("Skipping SFT training execution (DRY_RUN).")

## 💾 Step 7: Save and Package Trained Adapter

Once training completes, we save the resulting adapter weights and config, and bundle them into the `submission.zip` package.

In [ ]:
if SHOULD_RUN_PIPELINE:
    OUTPUT_DIR = "/kaggle/working"
    print(f"Saving trained adapter to {OUTPUT_DIR}...")
    trainer.model.save_pretrained(OUTPUT_DIR)

    # Compress the adapter files directly at the root of the archive
    zip_cmd = "zip -j submission.zip /kaggle/working/adapter_config.json /kaggle/working/adapter_model.safetensors"
    print(f"Executing: {zip_cmd}")
    subprocess.run(zip_cmd, shell=True, check=True)
    print("Trained submission.zip packaged successfully!")
else:
    print("Skipping adapter saving and packaging (DRY_RUN).")

In [ ]:
import shutil

# Keep your trained model weights, but wipe out heavy build artifacts
cache_dirs_to_clean = ["/kaggle/working/triton_cache", "/kaggle/working/torch_extensions"]

for cache_folder in cache_dirs_to_clean:
    if os.path.exists(cache_folder):
        print(f"Cleaning up temporary compilation cache: {cache_folder}")
        shutil.rmtree(cache_folder)
